PIFOCast, a simple weather prediction model
===========================================


##### Copyright 2025 Nicolas Gasnier.

In [1]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

##### Introduction

This project is based on the following :
- Google Deepmind [GraphCast](https://www.science.org/doi/10.1126/science.adi2336) and [GenCast](https://arxiv.org/abs/2312.15796) papers, and associated [source code](https://github.com/google-deepmind/graphcast) as a source of inspiration
- Tensorflow GNN's colab example ["Learning shortest path with GraphNetowks in TF-GNN"](https://colab.research.google.com/github/tensorflow/gnn/blob/master/examples/notebooks/graph_network_shortest_path.ipynb#scrollTo=qr1_8ttC08vu), from wich the base Encoder / Processor / Decoder architecture is derived.


In [2]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import math
import collections
import functools
import xarray as xr
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_gnn as tfgnn
from tensorflow_gnn import runner
from typing import Callable, Optional, Mapping

import importlib

# Theses two rows for improving development workflow of our module
import pifocast
importlib.reload(pifocast)

from pifocast import LatLonGrid, pifo, buildGridGNN, getGraphExample, getGraphForFeatures, get_dataset


2025-05-29 09:11:03.351651: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-29 09:11:03.540020: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-29 09:11:03.707613: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748502663.845733    9118 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748502663.884325    9118 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748502664.195903    9118 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

### Dataset generation - building the whole dataset from Era5

In [ ]:
schema = tfgnn.read_schema("pifo.pbtxt")
graph_spec = tfgnn.create_graph_spec_from_schema_pb(schema)
grid = LatLonGrid("dataset/202504.grib", 4)



In [ ]:
with tf.io.TFRecordWriter("dataset/inputDS.tfrecord") as inputDSWriter, tf.io.TFRecordWriter("dataset/targetDS.tfrecord") as targetDSWriter:
    for graph, target in pifoGridGenerator("dataset/202504.grib", grid):
        #graph, target = getGraphExample(grid)
        example = tfgnn.write_example(graph)
        inputDSWriter.write(example.SerializeToString())
        targetSR = tf.io.serialize_tensor(target)
        targetFeature = {
            'target' : tf.train.Feature(bytes_list=tf.train.BytesList(value=[targetSR.numpy()]))
        }    
        targetExample = tf.train.Example(features=tf.train.Features(feature=targetFeature))
        targetDSWriter.write(targetExample.SerializeToString())



In [ ]:
# TODO : tf.data.DataSet.zip(liste des datasets créés depuis TFRecord)
graphDS = tf.data.TFRecordDataset("dataset/inputDS.tfrecord")
resultDS = tf.data.TFRecordDataset("dataset/targetDS.tfrecord")
